In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

In [14]:
# Load data
X_train = pd.read_parquet('../data/X_train.parquet')

# Historical baseline: ONE row per series, plain aggregation, no expanding
def compute_historical_baseline(series):
    grouped = series.groupby(level='id')['value']
    return pd.DataFrame({
        'mean': grouped.mean(),
        'median': grouped.median(),
        'std': grouped.std(),
        'skew': grouped.skew(),
        'kurt': grouped.apply(lambda x: x.kurtosis()),
        'min': grouped.min(),
        'max': grouped.max(),
        'autocorr': grouped.apply(lambda x: x.autocorr(lag=1))
    })

historical_series = X_train.query("period == 1")
historical_baseline = compute_historical_baseline(historical_series)


# Online expanding stats: ONE row per (id, timestep) using only data seen so far
def compute_expanding_stats(series):
    grouped = series.groupby(level='id')['value']
    return pd.DataFrame({
        'mean': grouped.expanding().mean(),
        'median': grouped.expanding().median(),
        'std': grouped.expanding().std(),
        'skew': grouped.expanding().skew(),
        'kurt': grouped.expanding().apply(lambda x: x.kurtosis()),
        'min': grouped.expanding().min(),
        'max': grouped.expanding().max(),
        'autocorr': grouped.expanding().apply(lambda x: x.autocorr(lag=1))
    })

online_series = X_train.query("period == 2")

# Test on a small subset first (expanding + apply is expensive)
test_ids = online_series.index.get_level_values('id').unique()[:5]
online_subset = online_series.loc[test_ids]

online_expanding = compute_expanding_stats(online_subset)
online_expanding = online_expanding.droplevel(0)
online_expanding.head()

/Users/amine/miniconda3/envs/structural_break/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)


mean    median       std      skew      kurt       min       max  \
id time                                                                         
0  1192  0.078301  0.078301       NaN       NaN       NaN  0.078301  0.078301   
   1193 -0.419521 -0.419521  0.704027       NaN       NaN -0.917343  0.078301   
   1194 -1.193244 -0.917343  1.429604 -0.836112       NaN -2.740690  0.078301   
   1195 -1.493399 -1.655604  1.312587  0.413618 -3.067674 -2.740690  0.078301   
   1196 -0.918921 -0.917343  1.715310  0.319563 -1.557443 -2.740690  1.378991   

         autocorr  
id time            
0  1192       NaN  
   1193       NaN  
   1194  1.000000  
   1195  0.643608  
   1196 -0.173056